In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from mylibs import *
import os
import pandas as pd 

In [3]:
# Collect 520 products
list_links = []

driver = webdriver.Chrome()
driver.maximize_window()

for page in range(1, 14):
    url = f'https://hasaki.vn/danh-muc/cham-soc-da-mat-c4.html?p={page}'
    driver.get(url)

    # Locate the product grid
    product_container = driver.find_element(by=By.XPATH, value='//div[@class="ProductGrid__grid width_common"]')


    # Find the product detail links
    tag_links = product_container.find_elements(by=By.CSS_SELECTOR, value='.v3_thumb_common_sp.relative')
    for tag_link in tag_links:
        link = tag_link.get_attribute('href') + '\n'
        list_links.append(link)
print(f'Tìm thấy {len(list_links)} link sản phẩm')
write_txt(f'facial_skin_care_product_links.txt', list_links)

Tìm thấy 520 link sản phẩm


True

In [12]:

# Read the links and extract data from each product page
list_links = read_txt('facial_skin_care_product_links.txt')

for i in range(0, len(list_links)):
    link = list_links[i].rstrip('\n')
    driver.get(link)

    # 1. Product code
    tag_item_code = driver.find_element(by=By.XPATH, value='//span[@class="item-sku txt_color_1"]')
    item_code = tag_item_code.text.replace('Mã sản phẩm: ', '')
    print(item_code) 

    # 2. Product name
    tag_product_name = driver.find_element(by=By.CSS_SELECTOR, value='.product__title.base')
    product_name = tag_product_name.text
    print(product_name)

    # 3. Sale price
    tag_sale_price = driver.find_element(by=By.ID, value='product-final_price')
    sale_price = int(float(tag_sale_price.text.rstrip('₫').replace('.', '').strip()))
    print(sale_price)

    # 4. Original price
    try: 
        tag_origin_price = driver.find_element(by=By.ID, value='market_price')
        origin_price = int(float(tag_origin_price.text.rstrip('₫').replace('.', '').strip()))
        print(origin_price)
    except:
        origin_price = sale_price
        print(origin_price)
    # 5. Product variant
    tag_classify = driver.find_element(by=By.CLASS_NAME, value='product_chose_type')
    classify = tag_classify.text
    print(classify)

    # 6. Description
    tag_description = driver.find_element(by=By.ID, value='box_thongtinsanpham')
    description = tag_description.text
    print('Mô tả:', description)
    

    # 7. Average rating
    tag_point = driver.find_element(by=By.CSS_SELECTOR, value='.txt_numer.txt_color_2')
    point = tag_point.text
    print(point)


    # Save the record as JSON
    data_json_dir = f'data/facial_skin_care_products'
    if not os.path.exists(data_json_dir):
        os.makedirs(data_json_dir)

    no = i + 1
    file_name = f'{str(no).zfill(3)}.json'
    full_path = f'{data_json_dir}/{file_name}'
    data = {
        'Ma_san_pham': item_code,
        'Ten_san_pham': product_name,
        'Gia_ban': sale_price,
        'Gia_goc': origin_price,
        'Phan_loai': classify,
        'Mo_ta': description,
        'Diem_trung_binh': point
    }

    write_json(full_path, data)
    print(f'>>> Đã hoàn tất sản phẩm thứ {no}')
    print('-' * 50)

    
driver.close()


422218359
Sữa Chống Nắng Anessa Cho Da Nhạy Cảm & Trẻ Em 60ml (Mới)
534000
715000
Dung Tích:  60ml
Mô tả: Sữa Chống Nắng Anessa Perfect UV Sunscreen Mild Milk (For Sensitive Skin) SPF50+/PA++++ Cho Da Nhạy Cảm & Trẻ Em (Mới) là sản phẩm chống nắng da mặt đến từ thương hiệu Anessa - Nhật Bản. Sản phẩm với công nghệ Smooth Protect Air giúp phân tán bột chống tia UV thành bột nano có kích thước siêu nhỏ, nhờ đó, kết cấu trở nên mỏng nhẹ hơn, giảm tối đa tình trạng bết dính gây bí da, bảo vệ toàn diện làn da nhạy cảm.
Sữa Chống Nắng Cho Da Nhạy Cảm & Trẻ Em Anessa Perfect UV Sunscreen Mild Milk (For Sensitive Skin) SPF50+/PA++++ (Phiên bản mới) hiện đã có mặt tại Hasaki với 2 phân loại: 60ml, 60mlx2
    Làm sao để phân biệt hàng có trộn hay không ?
Hàng trộn sẽ không thể xuất hoá đơn đỏ (VAT) 100% được do có hàng không nguồn gốc trong đó.
Tại Hasaki, 100% hàng bán ra sẽ được xuất hoá đơn đỏ cho dù khách hàng có lấy hay không. Nếu có nhu cầu lấy hoá đơn đỏ, quý khách vui lòng lấy trước 22h 

In [4]:
# Combine the JSON records into a CSV file
list_item_codes = []
list_product_names = []
list_sale_prices = []
list_origin_prices = []
list_classifies = []
list_descriptions = []
list_points = []



data_dir = f'data/facial_skin_care_products'
for file_name in os.listdir(data_dir):
    if file_name.endswith('.json'):
        path = f'{data_dir}/{file_name}'
        dict_product = read_json(path)
        print(dict_product)

        # Append field values to their lists
        list_item_codes.append(dict_product['Ma_san_pham'])
        list_product_names.append(dict_product['Ten_san_pham'])
        list_sale_prices.append(dict_product['Gia_ban'])
        list_origin_prices.append(dict_product['Gia_goc'])
        list_classifies.append(dict_product['Phan_loai'])
        list_descriptions.append(dict_product['Mo_ta'])
        list_points.append(dict_product['Diem_trung_binh'])
else:

    # Export the data to CSV
    df_product = pd.DataFrame({
        'Ma_san_pham': list_item_codes,
        'Ten_san_pham': list_product_names,
        'Gia_ban': list_sale_prices,
        'Gia_goc': list_origin_prices,
        'Phan_loai': list_classifies,
        'Mo_ta': list_descriptions,
        'Diem_trung_binh': list_points
    })

    df_product.to_csv(f'products.csv', index=False)

{'Ma_san_pham': '318900012', 'Ten_san_pham': 'Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm 180ml', 'Gia_ban': 209000, 'Gia_goc': 435000, 'Phan_loai': 'Dung Tích: \n2x180ml\n180ml\nMùi hương:  Không Mùi', 'Mo_ta': 'Nước Hoa Hồng Klairs Supple Preparation là dòng sản phẩm toner được thương hiệu Dear, Klairs thiết kế chuyên biệt dành cho làn da nhạy cảm. Với bảng thành phần chiết xuất từ thực vật và kết cấu lỏng nhẹ, thấm nhanh trên da, nước hoa hồng Klairs sẽ giúp cân bằng độ pH và cấp ẩm cho làn da hiệu quả, hỗ trợ cho các bước skincare tiếp theo đạt hiệu quả tối ưu.\nHiện sản phẩm Nước Hoa Hồng Klairs Cho Da Nhạy Cảm 180ml đã có mặt tại Hasaki với 2 loại:\nNước Hoa Hồng Klairs Supple Preparation Facial Toner: dành cho da nhạy cảm (hương thảo mộc).\nNước Hoa Hồng Klairs Supple Preparation Unscented Toner: dành cho da siêu nhạy cảm (không mùi).\n1. Nước hoa hồng Dear, Klairs Supple Preparation Facial Toner 180ml\nNước hoa hồng Dear, Klairs Supple Preparation Facial Toner sản phẩm tối ư